# D11 · Modbus RTU over RS-485 and the half-duplex time budget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D11_modbus_rtu_over_rs_485_and_the_half_duplex_time_.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 55 min

**Skill Hub concepts:** Modbus RTU framing; RS-485 driver-enable timing; Half-duplex bus time budget

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D11")

## Map: one master, many slaves, one pair of wires

```
 frame:  [slave addr][function][data ...][CRC lo][CRC hi]      CRC-16/MODBUS, low byte first
 gap:    >= 3.5 character times of silence ends a frame (fixed 1.75 ms above 19200 baud)
 char:   11 bits = start + 8 data + parity + stop   (or 2 stop bits when no parity)
```

| Function | Request | Normal response | Exception response |
|---|---|---|---|
| 0x03 Read holding registers | addr, 03, start(2), count(2), CRC | addr, 03, byte count, data(2 x count), CRC | addr, 0x83, code, CRC |
| 0x06 Write single register | addr, 06, reg(2), value(2), CRC | echo of the request | addr, 0x86, code, CRC |

Exception codes: 01 illegal function, 02 illegal data address, 03 illegal data value.

**RS-485 driver enable (DE):** the transceiver drives the bus only while DE is high. Raise DE before the first start
bit; drop it after the **last stop bit has left the shift register**: that is the UART's TC (transmission complete)
flag, not TXE (transmit data register empty), which fires one character early.

**Half-duplex budget:** one poll costs
$t = \dfrac{(N_{req} + N_{resp} + 2 \times 3.5)\, b}{\text{baud}} + t_{turnaround}$, with $b$ bits per character.

In [ ]:
def crc16_modbus(data: bytes) -> int:      # from D01, given
    crc = 0xFFFF
    for b in data:
        crc ^= b
        for _ in range(8):
            crc = (crc >> 1) ^ 0xA001 if crc & 1 else crc >> 1
    return crc

class ModbusException(Exception):
    pass

## Task 1: master side

In [ ]:
def read_holding_request(slave, start, count):
    raise NotImplementedError  # TODO

def parse_read_response(frame, slave):
    """Return the list of register values; raise ValueError on a bad CRC or address,
    ModbusException(code) on an exception response."""
    raise NotImplementedError  # TODO

lab.check("master", read_holding_request, parse_read_response)

In [ ]:
%%writefile d11_master.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d11.h
// Grade it with lab.check("master", "d11_master.c"). For C++, name the file .cpp in both places.
#include "d11.h"

size_t mb_read_holding_request(uint8_t slave, uint16_t start, uint16_t count, uint8_t *out) {
    (void)slave; (void)start; (void)count; (void)out;
    return 0; /* TODO: address, 0x03, start and count big-endian, CRC little-endian */
}

int mb_parse_read_response(const uint8_t *frame, size_t len, uint8_t slave, uint16_t *regs, size_t regs_cap, uint8_t *exception_code) {
    (void)frame; (void)len; (void)slave; (void)regs; (void)regs_cap; (void)exception_code;
    return MB_ERR_FRAME; /* TODO */
}

In [ ]:
# lab.check("master", "d11_master.c")  # uncomment after writing the C/C++ cell above

## Task 2: slave side
`regs` is a dict `{address: value}`. Support 0x03 and 0x06; reply with exception 01 for other functions and 02 for
addresses not in `regs`. Return `None` if the CRC is bad or the frame is for another slave (a slave stays silent).

In [ ]:
def slave_handle(frame, my_addr, regs):
    raise NotImplementedError  # TODO

lab.check("slave", slave_handle)

In [ ]:
%%writefile d11_slave.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d11.h
// Grade it with lab.check("slave", "d11_slave.c"). For C++, name the file .cpp in both places.
#include "d11.h"

size_t mb_slave_handle(const uint8_t *frame, size_t len, uint8_t my_addr, MbRegister *regs, size_t n_regs, uint8_t *resp, size_t resp_cap) {
    (void)frame; (void)len; (void)my_addr; (void)regs; (void)n_regs; (void)resp; (void)resp_cap;
    return 0; /* TODO: check CRC and address; 0x03 read, 0x06 write (echo), else exception 0x01 */
}

In [ ]:
# lab.check("slave", "d11_slave.c")  # uncomment after writing the C/C++ cell above

## Task 3: the budget

In [ ]:
def poll_time(baud, req_bytes, resp_bytes, turnaround_s, bits_per_char=11):
    raise NotImplementedError  # TODO

def max_poll_rate_per_node(n_nodes, baud, req_bytes, resp_bytes, turnaround_s):
    raise NotImplementedError  # TODO

lab.check("budget", poll_time, max_poll_rate_per_node)

In [ ]:
%%writefile d11_budget.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d11.h
// Grade it with lab.check("budget", "d11_budget.c"). For C++, name the file .cpp in both places.
#include "d11.h"

double mb_poll_time(double baud, int req_bytes, int resp_bytes, double turnaround_s, int bits_per_char) {
    (void)baud; (void)req_bytes; (void)resp_bytes; (void)turnaround_s; (void)bits_per_char;
    return 0; /* TODO: include the 3.5-character silences (7 characters per poll) */
}

double mb_max_poll_rate_per_node(int n_nodes, double baud, int req_bytes, int resp_bytes, double turnaround_s) {
    (void)n_nodes; (void)baud; (void)req_bytes; (void)resp_bytes; (void)turnaround_s;
    return 0; /* TODO */
}

In [ ]:
# lab.check("budget", "d11_budget.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** 3.5 character times at 9600 baud with 11-bit characters, in milliseconds (2 decimals)?

**M2 (Apply).** Response length in bytes when reading 10 holding registers?

**M3 (Analyze).** One poll at 19200 baud: 8-byte request, 25-byte response, 1 ms slave turnaround. Milliseconds?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** The last byte of every reply arrives corrupted at the master. The slave drops DE in the TXE
interrupt. Fix?
- a) raise the baud rate
- b) drop DE in the TC (transmission complete) interrupt, after the stop bit has left the shift register
- c) add a 120 ohm resistor to DE

**K2 (Evaluate).** Eight smart servos share an RS-485 bus at 1 Mbaud, each polled for position and sent a new
setpoint. Estimate whether a 500 Hz control loop fits, list what eats the budget, and say why the setpoint command
should be idempotent.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d11.h`).
3. In `roboprep/checks/d11.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()